# Named latent dimensions: scientific equivalence

Freeze one exact OPD cube, then compare explicit single-screen propagation with named batching. Wavelength remains at -3. No output is renormalized. Counts use seconds weights and pixel area once. Run from a clean kernel after installing `fiatlux[tutorials]`.

In [ ]:
import json
import torch
from fiatlux import (
    Grid, Spectrum, PhotometricBand, PlaneWave, FieldDimension, SerialSystem,
    FFTPropagator, MFTPropagator, ZeldaMask, ZeldaStop, DeformableMirror,
    ActuatorGrid, ZernikeBasis, KolmogorovAtmosphereModel, Detector,
    ExposureAccumulator, Atmosphere, FatmossAtmosphereModel,
)

torch.set_num_threads(1)
dtype = torch.float64
grid = Grid(32, 32, .02, .02, dtype=dtype)
focal_grid = Grid(32, 32, 1e-6, 1e-6, dtype=dtype)
mono = PlaneWave(Spectrum(0, PhotometricBand.HCM2, 1, dtype=dtype)).generate_field(grid)
poly = PlaneWave(Spectrum(0, PhotometricBand.HCM2, 3, dtype=dtype)).generate_field(grid)
model = KolmogorovAtmosphereModel(grid, .2, seed=902)
opd = model.sample_opd_many(8)  # Freeze once; both paths use exactly these values.
time = FieldDimension('time', 8, torch.arange(8, dtype=dtype)*.01,
                      's', torch.full((8,), .01, dtype=dtype))
dm = DeformableMirror(grid, ActuatorGrid(3, 3, .1), grid, ZernikeBasis(grid, 3))
dm.commands = torch.tensor([20e-9, -10e-9, 5e-9], dtype=dtype)
cases = [
    ('FFT Fraunhofer', mono, [FFTPropagator(2.)]),
    ('MFT polychromatic Fraunhofer', poly, [MFTPropagator(2., focal_grid)]),
    ('MFT Fresnel', mono, [MFTPropagator(output_grid=focal_grid, propagation='fresnel', distance=2.)]),
    ('ZELDA', mono, [MFTPropagator(2., focal_grid), ZeldaMask(focal_grid, 1e-6, 2.5e-7),
                     MFTPropagator(-2., grid), ZeldaStop(grid, .25)]),
    ('Static DM + atmosphere', mono, [dm, FFTPropagator(2.)]),
]


def errors(actual, expected):
    delta = (actual-expected).abs()
    # Raw maximum relative error can be sensitive to near-zero samples; retain
    # the absolute error alongside it. This does not rescale either result.
    return {'max_absolute': float(delta.max()),
            'max_relative': float((delta/expected.abs().clamp_min(torch.finfo(dtype).tiny)).max()),
            'rms': float(delta.square().mean().sqrt())}


rows = []
images = {}
with torch.no_grad():
    for name, source_field, elements in cases:
        system = SerialSystem(elements)
        loop = torch.stack([system.run_field(source_field.apply_opd(screen)).final_field.complex_amplitude for screen in opd])
        buffer = source_field.apply_opd(opd, dimensions=(time,))
        batched = system.run_field(buffer).final_field
        torch.testing.assert_close(batched.complex_amplitude, loop, rtol=1e-11, atol=1e-8)
        expected = (loop.abs().square().sum(-3)*.01).sum(0)*batched.grid.dx*batched.grid.dy
        detector = Detector(batched.grid, exposure_time=.08)
        one_shot = detector.acquire(batched, integrate_over='time')
        accumulator = ExposureAccumulator(detector)
        for start, stop in [(0, 3), (3, 6), (6, 8)]:
            accumulator.add(batched.slice('time', start, stop))
        chunked = accumulator.finish()
        torch.testing.assert_close(one_shot, expected, rtol=1e-11, atol=1e-8)
        torch.testing.assert_close(chunked, expected, rtol=1e-11, atol=1e-8)
        images[name] = one_shot
        rows.append({'case': name, 'opd_shape': list(opd.shape),
                     'input_field_shape': list(buffer.complex_amplitude.shape),
                     'output_field_shape': list(batched.complex_amplitude.shape), 'field': errors(batched.complex_amplitude, loop),
                     'exposure': errors(one_shot, expected), 'chunked': errors(chunked, expected)})
print(json.dumps(rows, indent=2))


## Atmosphere buffer and long-exposure image

The table above reports maximum absolute, maximum pointwise relative and RMS
errors for complex amplitudes and counts. Relative errors near dark pixels may
be large; absolute errors and the dtype-dependent tests remain essential.
Amplitudes use sqrt(photons/s/m²); with QE=1 the final counts are electrons/pixel.
No amplitude or image has been renormalized. Each regular time weight is dt;
`exposure_time=T*dt` is checked, never multiplied a second time.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, sample in zip(axes[:2], [0, 1]):
    im = ax.imshow(opd[sample].numpy()*1e9, origin='lower',
                   vmin=float(opd.min()*1e9), vmax=float(opd.max()*1e9))
    ax.set_title(f'Frozen OPD sample {sample}')
    fig.colorbar(im, ax=ax, label='OPD [nm]')
im = axes[2].imshow(images['FFT Fraunhofer'].numpy(), origin='lower')
axes[2].set_title('Long exposure: T=8, dt=0.01 s')
fig.colorbar(im, ax=axes[2], label='Electrons / pixel (QE=1)')
for ax in axes:
    ax.set_xlabel('x pixel')
    ax.set_ylabel('y pixel')
plt.show()


### Incoherent exposure is not a coherent average

Two equal fields with opposite phase cancel when amplitudes are averaged.
Their intensities still add at the detector. This deliberately simple check
also catches an accidental second exposure-time multiplier.


In [ ]:
two_times = FieldDimension('time', 2, unit='s',
                           integration_weights=torch.full((2,), .01, dtype=dtype))
opposite = mono.expand_dimension(two_times) * torch.tensor([1., -1.], dtype=dtype)[:, None, None, None]
assert opposite.coherent_mean('time').intensity().sum() == 0
expected = mono.intensity().sum(-3)*.02*grid.dx*grid.dy
counts = Detector(grid, exposure_time=.02).acquire(opposite, integrate_over='time')
torch.testing.assert_close(counts, expected, rtol=1e-11, atol=1e-8)
assert counts.sum() > 0
print('Opposite-phase exposure errors:', errors(counts, expected))


## Correlated temporal buffers

An injected, seeded translating backend tests the adapter without requiring the optional FATMOSS dependency. The real backend remains covered by optional tutorial 10. Both chunk partitions must integrate the same exposure, and `advance=False` must preserve selected time.

In [ ]:
# Injected backend: optional FATMOSS installation is not needed for this check.
import numpy as np

class TranslatingScreen:
    def __init__(self, seed=51):
        self.phase = np.random.default_rng(seed).uniform(0, 2*np.pi)
    def GetScreenByTimestep(self, timestep):
        x = np.arange(32)[:, None] - .37*timestep  # non-integer pixel displacement
        y = np.arange(32)[None, :]
        return 80*np.cos(2*np.pi*x/32 + self.phase) * np.cos(2*np.pi*y/32)  # nm, (x,y)

backend = TranslatingScreen()
model = FatmossAtmosphereModel(grid, backend, time_step=.01)
# Record the exact cube consumed by apply_buffer; the reference never resamples.
sampled_cubes = []
sequence_opd = model.sequence_opd

def record_sequence(*args, **kwargs):
    cube = sequence_opd(*args, **kwargs)
    sampled_cubes.append(cube.clone())
    return cube

model.sequence_opd = record_sequence
atmosphere = Atmosphere(grid, model)
buffer = atmosphere.apply_buffer(mono, 8, advance=False)
assert model.current_time == 0
propagator = FFTPropagator(2.)
with torch.no_grad():
    propagated = propagator.apply(buffer)
    cube = sampled_cubes[0]
    loop = torch.stack([propagator.apply(mono.apply_opd(screen)).complex_amplitude for screen in cube])
    torch.testing.assert_close(propagated.complex_amplitude, loop)
    detector = Detector(propagated.grid, exposure_time=.08)
    expected = (loop.abs().square().sum(-3)*.01).sum(0)*propagated.grid.dx*propagated.grid.dy
    integrated = detector.acquire(propagated, integrate_over='time')
    torch.testing.assert_close(integrated, expected, rtol=1e-11, atol=1e-8)
    partitions = []
    for lengths in ([1, 3, 4], [4, 4]):
        model.seek(0)
        accumulator = ExposureAccumulator(detector)
        for length in lengths:
            accumulator.add(propagator.apply(atmosphere.apply_buffer(mono, length)))
        result = accumulator.finish()
        torch.testing.assert_close(result, expected)
        assert model.current_time == .08
        partitions.append({'lengths': lengths, **errors(result, expected)})
print(json.dumps({'FATMOSS injected backend': errors(propagated.complex_amplitude, loop),
                  'exposure': errors(integrated, expected), 'partitions': partitions}, indent=2))

# Scalar replay and same-seed replay are separate from the optical comparison.
scalar_cube = torch.stack([model.opd_at(t*.01) for t in range(8)])
torch.testing.assert_close(scalar_cube, cube, rtol=0, atol=0)
replay_model = FatmossAtmosphereModel(grid, TranslatingScreen(seed=51), time_step=.01)
torch.testing.assert_close(replay_model.sequence_opd(8, advance=False), cube, rtol=0, atol=0)
from fiatlux.optics.turbulence_validation import temporal_autocorrelation
torch.testing.assert_close(temporal_autocorrelation(scalar_cube),
                           temporal_autocorrelation(cube), rtol=0, atol=0)
assert model.current_time == .08  # Replay queries did not advance the selected time.


## Memory and performance

Use `ExposureAccumulator(..., track_grad=False)` for detached, bounded accumulator memory. Full gradient tracking retains chunk graphs. Run `python benchmarks/latent_dimensions.py --output results.json` for CPU/CUDA timing, errors and memory; no timing assertion belongs in CI. Shack–Hartmann currently rejects latent fields explicitly.